# ALS Boundary Sensitivity Stage-1 (Cross-Lingual Layer Composition)

## Part 0. Title and experiment purpose

**Research question:** Is cross-lingual layer composition sensitive to the choice of composition boundary?

This notebook is a **Stage-1 identification study**. It is designed to quantify boundary sensitivity through:
- deltas versus a no-swap baseline,
- reproducible ranking differences across boundaries,
- stable vs unstable boundary regimes.

### What this notebook does
- Runs **post-hoc hard layer composition** only (no new training).
- Compares:
  1. No-swap baseline,
  2. Bandarkar-style reference composition (simplified reference, not exact reproduction),
  3. Boundary-indexed hard-swap conditions.
- Produces a boundary performance map and stability statistics.

### What this notebook does **not** do
- No DANN/adversarial/alignment/LoRA training.
- No mechanistic causality claims.
- No "best recipe" optimization framing.


## Part 1. Environment and imports


In [ ]:
# Optional install helper (uncomment if needed in a fresh environment)
# %pip install -q transformers datasets accelerate sentencepiece pandas numpy matplotlib seaborn scipy

import copy
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import platform
import random
import re
import sys
from typing import Any, Callable, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from torch import nn

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    set_seed as hf_set_seed,
)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("Datasets:", __import__("datasets").__version__)
print("CUDA available:", torch.cuda.is_available())


## Part 2. Global config


In [ ]:
@dataclass
class ExperimentConfig:
    mode: str
    model_family: str
    target_model_name: str
    anchor_model_name: str
    use_instruct: bool
    dataset_name: str
    dataset_config_name: str
    dataset_split: str
    language: str
    max_eval_samples: Optional[int]
    boundaries_full: List[int]
    boundaries_sanity: List[int]
    upper_boundary_t: int
    seeds: List[int]
    max_new_tokens: int
    do_sample: bool
    temperature: float
    top_p: float
    num_beams: int
    stop_strings: List[str]
    device: str
    torch_dtype: str
    output_root: str
    run_name_prefix: str


def build_experiment_config() -> ExperimentConfig:
    mode = os.environ.get("ALS_MODE", "sanity").strip().lower()
    if mode not in {"sanity", "full"}:
        raise ValueError(f"ALS_MODE must be 'sanity' or 'full', got: {mode}")

    use_instruct = True
    model_name = "Qwen/Qwen2.5-1.5B-Instruct" if use_instruct else "Qwen/Qwen2.5-1.5B"

    return ExperimentConfig(
        mode=mode,
        model_family="qwen2.5",
        target_model_name=model_name,
        anchor_model_name=model_name,  # TODO: replace with a task/language anchor checkpoint.
        use_instruct=use_instruct,
        dataset_name="juletxara/mgsm",
        dataset_config_name="te",
        dataset_split="test",
        language="te",
        max_eval_samples=32 if mode == "sanity" else None,
        boundaries_full=[4, 8, 12, 16],
        boundaries_sanity=[4, 12],
        upper_boundary_t=20,
        seeds=[13, 42, 2026],
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        num_beams=1,
        stop_strings=[],
        device="cuda" if torch.cuda.is_available() else "cpu",
        torch_dtype="float16" if torch.cuda.is_available() else "float32",
        output_root="outputs",
        run_name_prefix="ALS_boundary_sensitivity_stage1",
    )

cfg = build_experiment_config()
print(cfg)


## Part 3. Validation and sanity checks


In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    hf_set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def validate_boundary_grid(cfg: ExperimentConfig, model_config: AutoConfig) -> List[int]:
    boundaries = cfg.boundaries_sanity if cfg.mode == "sanity" else cfg.boundaries_full
    L = int(getattr(model_config, "num_hidden_layers", -1))
    if L <= 0:
        raise ValueError("Could not infer num_hidden_layers from model config.")
    t = cfg.upper_boundary_t
    if not (0 < t < L):
        raise ValueError(f"Invalid upper boundary t={t} for model depth L={L}.")
    for b in boundaries:
        if not (0 <= b < t < L):
            raise ValueError(f"Invalid boundary b={b}. Must satisfy 0 <= b < t={t} < L={L}.")
    return boundaries


def validate_model_compatibility(target_model, anchor_model) -> None:
    tc, ac = target_model.config, anchor_model.config
    checks = {
        "model_type": (tc.model_type, ac.model_type),
        "num_hidden_layers": (tc.num_hidden_layers, ac.num_hidden_layers),
        "hidden_size": (tc.hidden_size, ac.hidden_size),
        "num_attention_heads": (tc.num_attention_heads, ac.num_attention_heads),
        "vocab_size": (tc.vocab_size, ac.vocab_size),
    }
    mismatches = {k: v for k, v in checks.items() if v[0] != v[1]}
    if mismatches:
        raise ValueError(f"Target/anchor incompatibility: {mismatches}")


def validate_tokenizer_compatibility(target_tokenizer, anchor_tokenizer) -> None:
    if target_tokenizer.get_vocab() != anchor_tokenizer.get_vocab():
        raise ValueError("Tokenizer vocab mismatch.")


def validate_dataset(dataset: pd.DataFrame) -> None:
    required_cols = ["sample_id", "prompt", "gold_answer"]
    missing = [c for c in required_cols if c not in dataset.columns]
    if missing:
        raise ValueError(f"Dataset missing columns: {missing}")
    if dataset["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample_id values found.")
    if len(dataset) == 0:
        raise ValueError("Dataset is empty.")


def validate_answer_parser(parser_fn: Callable[[str], Dict[str, Any]], sample_batch: List[str]) -> None:
    for s in sample_batch:
        out = parser_fn(s)
        for key in ["parsed_answer", "is_valid", "error_type"]:
            if key not in out:
                raise ValueError(f"Parser missing key: {key}")


## Part 4. Shared helper functions


In [ ]:
def load_mgsm_te_dataset(cfg: ExperimentConfig) -> pd.DataFrame:
    try:
        ds = load_dataset(cfg.dataset_name, cfg.dataset_config_name, split=cfg.dataset_split)
    except Exception as e:
        raise RuntimeError("Failed to load MGSM Telugu dataset.") from e

    q_col = next((c for c in ["question", "problem", "prompt"] if c in ds.column_names), None)
    a_col = next((c for c in ["answer", "final_answer", "target", "label"] if c in ds.column_names), None)
    if q_col is None or a_col is None:
        raise ValueError(f"Could not resolve columns from {ds.column_names}")

    rows = []
    for i, ex in enumerate(ds):
        rows.append({
            "sample_id": str(ex.get("id", f"te_{i:05d}")),
            "prompt": str(ex[q_col]).strip(),
            "gold_answer": str(ex[a_col]).strip(),
            "metadata": {k: ex[k] for k in ex.keys() if k not in {q_col, a_col}},
        })
    df = pd.DataFrame(rows).sort_values("sample_id", kind="mergesort").reset_index(drop=True)
    if cfg.max_eval_samples is not None:
        df = df.iloc[: cfg.max_eval_samples].copy()
    validate_dataset(df)
    return df


def normalize_numeric_string(text: str) -> str:
    return re.sub(r"\s+", "", text.strip().replace(",", ""))


def extract_final_answer(raw_output: str) -> Dict[str, Any]:
    txt = raw_output.strip()
    if txt == "":
        return {"parsed_answer": None, "is_valid": False, "error_type": "empty_output"}

    m = re.search(r"(?i)(final\s*answer\s*[:：]\s*)([-+]?\d[\d,\.]*)", txt)
    if m:
        return {"parsed_answer": normalize_numeric_string(m.group(2)), "is_valid": True, "error_type": None}

    nums = re.findall(r"[-+]?\d[\d,\.]*", txt)
    if nums:
        return {"parsed_answer": normalize_numeric_string(nums[-1]), "is_valid": True, "error_type": None}

    return {"parsed_answer": None, "is_valid": False, "error_type": "no_numeric_answer"}


def format_prompt_for_mgsm(question: str) -> str:
    return (
        "Solve the following math word problem. Show brief reasoning and end with "
        "'Final Answer: <number>'.\n\n"
        f"Problem:\n{question}\n"
    )


def compute_sample_record(**kwargs) -> Dict[str, Any]:
    gold = normalize_numeric_string(str(kwargs["gold_answer"]))
    pred = kwargs["parsed_answer"]
    pred = None if pred is None else normalize_numeric_string(str(pred))
    return {
        "run_id": kwargs["run_id"],
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "seed": kwargs["seed"],
        "condition_name": kwargs["condition_name"],
        "boundary": kwargs["boundary"],
        "sample_id": kwargs["sample_id"],
        "prompt": kwargs["prompt"],
        "raw_output": kwargs["raw_output"],
        "parsed_answer": pred,
        "gold_answer": gold,
        "is_valid": kwargs["is_valid"],
        "is_correct": bool(kwargs["is_valid"] and pred == gold),
        "error_type": kwargs["error_type"],
        "response_length": len(kwargs["raw_output"]),
        "model_family": kwargs["model_family"],
        "target_model_name": kwargs["target_model_name"],
        "anchor_model_name": kwargs["anchor_model_name"],
    }


## Part 5. Model loading


In [ ]:
def _dtype_from_cfg(cfg):
    return torch.float16 if cfg.torch_dtype == "float16" else torch.float32


def load_target_model_and_tokenizer(cfg):
    tok = AutoTokenizer.from_pretrained(cfg.target_model_name, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        cfg.target_model_name,
        torch_dtype=_dtype_from_cfg(cfg),
        device_map="auto" if cfg.device == "cuda" else None,
    )
    if cfg.device != "cuda":
        model.to(cfg.device)
    model.eval()
    return model, tok


def load_anchor_model_and_tokenizer(cfg):
    tok = AutoTokenizer.from_pretrained(cfg.anchor_model_name, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        cfg.anchor_model_name,
        torch_dtype=_dtype_from_cfg(cfg),
        device_map="auto" if cfg.device == "cuda" else None,
    )
    if cfg.device != "cuda":
        model.to(cfg.device)
    model.eval()
    return model, tok


## Part 6. Baseline and composition builders


In [ ]:
def _get_transformer_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    raise ValueError("Unsupported model architecture for layer extraction.")


def clone_model_structure(model):
    return copy.deepcopy(model)


def build_no_swap_view(target_model):
    return clone_model_structure(target_model)


def build_bandarkar_reference_view(target_model, anchor_model, *, lower_boundary, upper_boundary, enable_transition=False):
    if enable_transition:
        raise NotImplementedError("Transition disabled for Stage-1.")
    composed = clone_model_structure(target_model)
    a_layers = _get_transformer_layers(anchor_model)
    c_layers = _get_transformer_layers(composed)
    L = len(c_layers)
    if not (0 <= lower_boundary < upper_boundary < L):
        raise ValueError(f"Invalid boundaries: lower={lower_boundary}, upper={upper_boundary}, L={L}")
    for i in range(lower_boundary, upper_boundary):
        c_layers[i] = copy.deepcopy(a_layers[i])
    composed.eval()
    return composed


def build_hard_swap_view(target_model, anchor_model, *, boundary_b, boundary_t):
    return build_bandarkar_reference_view(
        target_model, anchor_model, lower_boundary=boundary_b, upper_boundary=boundary_t, enable_transition=False
    )


## Part 7. Evaluation runner


In [ ]:
@torch.inference_mode()
def generate_one(model, tokenizer, prompt, gen_cfg, device):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=gen_cfg.max_new_tokens,
        do_sample=gen_cfg.do_sample,
        temperature=gen_cfg.temperature,
        top_p=gen_cfg.top_p,
        num_beams=gen_cfg.num_beams,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def evaluate_condition_on_dataset(*, cfg, run_id, seed, condition_name, boundary, model, tokenizer, dataset_df):
    set_all_seeds(seed)
    gen_cfg = GenerationConfig(
        max_new_tokens=cfg.max_new_tokens,
        do_sample=cfg.do_sample,
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        num_beams=cfg.num_beams,
    )
    records = []
    for _, row in dataset_df.iterrows():
        prompt = format_prompt_for_mgsm(row["prompt"])
        raw = generate_one(model, tokenizer, prompt, gen_cfg, cfg.device)
        parsed = extract_final_answer(raw)
        records.append(compute_sample_record(
            run_id=run_id, seed=seed, condition_name=condition_name, boundary=boundary,
            sample_id=row["sample_id"], prompt=prompt, raw_output=raw,
            parsed_answer=parsed["parsed_answer"], gold_answer=row["gold_answer"],
            is_valid=parsed["is_valid"], error_type=parsed["error_type"],
            model_family=cfg.model_family, target_model_name=cfg.target_model_name, anchor_model_name=cfg.anchor_model_name,
        ))
    return pd.DataFrame(records)


## Part 8. Main boundary scan


In [ ]:
def make_run_id(cfg):
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    digest = hashlib.md5(json.dumps(asdict(cfg), sort_keys=True).encode()).hexdigest()[:8]
    return f"{cfg.run_name_prefix}_{cfg.mode}_{ts}_{digest}"

cfg = build_experiment_config()
run_id = make_run_id(cfg)
output_dir = Path(cfg.output_root) / run_id
plots_dir = output_dir / "plots"
output_dir.mkdir(parents=True, exist_ok=False)
plots_dir.mkdir(parents=True, exist_ok=True)

# Load and validate
target_model, target_tok = load_target_model_and_tokenizer(cfg)
anchor_model, anchor_tok = load_anchor_model_and_tokenizer(cfg)
validate_model_compatibility(target_model, anchor_model)
validate_tokenizer_compatibility(target_tok, anchor_tok)
active_boundaries = validate_boundary_grid(cfg, target_model.config)
dataset_df = load_mgsm_te_dataset(cfg)
validate_answer_parser(extract_final_answer, ["Final Answer: 42", "answer 1,234", "", "no number"])

dataset_df[["sample_id", "prompt"]].to_csv(output_dir / "evaluated_samples.csv", index=False)

all_records = []
for seed in cfg.seeds:
    model_ns = build_no_swap_view(target_model)
    all_records.append(evaluate_condition_on_dataset(
        cfg=cfg, run_id=run_id, seed=seed, condition_name="no_swap", boundary=None,
        model=model_ns, tokenizer=target_tok, dataset_df=dataset_df,
    ))
    del model_ns

    ref_b = min(active_boundaries)
    model_ref = build_bandarkar_reference_view(
        target_model, anchor_model, lower_boundary=ref_b, upper_boundary=cfg.upper_boundary_t, enable_transition=False
    )
    all_records.append(evaluate_condition_on_dataset(
        cfg=cfg, run_id=run_id, seed=seed, condition_name="bandarkar_style_reference", boundary=ref_b,
        model=model_ref, tokenizer=target_tok, dataset_df=dataset_df,
    ))
    del model_ref

    for b in active_boundaries:
        model_hs = build_hard_swap_view(target_model, anchor_model, boundary_b=b, boundary_t=cfg.upper_boundary_t)
        all_records.append(evaluate_condition_on_dataset(
            cfg=cfg, run_id=run_id, seed=seed, condition_name="hard_swap", boundary=b,
            model=model_hs, tokenizer=target_tok, dataset_df=dataset_df,
        ))
        del model_hs

sample_records_df = pd.concat(all_records, ignore_index=True)
sample_records_df.head(2)


## Part 9. Aggregation and analysis


In [ ]:
def aggregate_sample_records(df):
    out = (
        df.groupby(["run_id", "seed", "condition_name", "boundary"], dropna=False)
          .agg(
              n_samples=("sample_id", "count"),
              accuracy=("is_correct", "mean"),
              valid_rate=("is_valid", "mean"),
              malformed_rate=("is_valid", lambda x: 1.0 - float(np.mean(x))),
              empty_rate=("error_type", lambda s: float((s == "empty_output").mean())),
              mean_response_length=("response_length", "mean"),
          )
          .reset_index()
    )
    return out


def attach_delta_vs_noswap(summary_df):
    base = summary_df[summary_df["condition_name"] == "no_swap"][["seed", "accuracy", "valid_rate"]]
    base = base.rename(columns={"accuracy": "base_accuracy", "valid_rate": "base_valid_rate"})
    merged = summary_df.merge(base, on="seed", how="left", validate="many_to_one")
    merged["delta_accuracy_vs_noswap"] = merged["accuracy"] - merged["base_accuracy"]
    merged["delta_valid_rate_vs_noswap"] = merged["valid_rate"] - merged["base_valid_rate"]
    return merged


def compute_boundary_rank_stability(summary_df):
    hs = summary_df[summary_df["condition_name"] == "hard_swap"].copy()
    hs["boundary"] = hs["boundary"].astype(int)
    hs["rank_within_seed"] = hs.groupby("seed")["accuracy"].rank(method="average", ascending=False)
    stab = (
        hs.groupby("boundary")
          .agg(
              mean_accuracy=("accuracy", "mean"),
              std_accuracy=("accuracy", "std"),
              mean_delta_accuracy=("delta_accuracy_vs_noswap", "mean"),
              mean_rank=("rank_within_seed", "mean"),
              std_rank=("rank_within_seed", "std"),
          )
          .reset_index()
          .sort_values("mean_rank", ascending=True)
    )
    return hs, stab


def select_stable_and_unstable_boundaries(stability_df):
    if len(stability_df) < 2:
        return {"stable_boundaries": [], "unstable_boundaries": [], "rule": "insufficient_boundaries"}
    q_rank_lo, q_rank_hi = stability_df["std_rank"].quantile([0.25, 0.75]).tolist()
    q_acc_lo, q_acc_hi = stability_df["std_accuracy"].quantile([0.25, 0.75]).tolist()
    stable = stability_df[(stability_df["std_rank"] <= q_rank_lo) & (stability_df["std_accuracy"] <= q_acc_lo)]["boundary"].tolist()
    unstable = stability_df[(stability_df["std_rank"] >= q_rank_hi) & (stability_df["std_accuracy"] >= q_acc_hi)]["boundary"].tolist()
    return {
        "stable_boundaries": [int(x) for x in stable],
        "unstable_boundaries": [int(x) for x in unstable],
        "rule": {
            "std_rank_q25": q_rank_lo,
            "std_rank_q75": q_rank_hi,
            "std_accuracy_q25": q_acc_lo,
            "std_accuracy_q75": q_acc_hi,
        },
    }

summary_by_seed = attach_delta_vs_noswap(aggregate_sample_records(sample_records_df))
hard_swap_by_seed, boundary_stability = compute_boundary_rank_stability(summary_by_seed)
selected_boundaries = select_stable_and_unstable_boundaries(boundary_stability)
summary_across_seeds = (
    summary_by_seed.groupby(["condition_name", "boundary"], dropna=False)
      .agg(
          mean_accuracy=("accuracy", "mean"),
          std_accuracy=("accuracy", "std"),
          mean_valid_rate=("valid_rate", "mean"),
          mean_malformed_rate=("malformed_rate", "mean"),
          mean_delta_accuracy_vs_noswap=("delta_accuracy_vs_noswap", "mean"),
          mean_delta_valid_rate_vs_noswap=("delta_valid_rate_vs_noswap", "mean"),
      ).reset_index()
)
summary_by_seed.head()


## Part 10. Visualization


In [ ]:
def plot_boundary_accuracy(summary_by_seed, out_path):
    hs = summary_by_seed[summary_by_seed["condition_name"] == "hard_swap"].copy()
    hs["boundary"] = hs["boundary"].astype(int)
    plt.figure(figsize=(7, 4))
    sns.lineplot(data=hs, x="boundary", y="accuracy", hue="seed", marker="o", alpha=0.6)
    mean_df = hs.groupby("boundary", as_index=False)["accuracy"].mean()
    sns.lineplot(data=mean_df, x="boundary", y="accuracy", color="black", marker="s", linewidth=2.5, label="mean")
    plt.title("Boundary Performance Map: Accuracy")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def plot_delta_vs_noswap(summary_by_seed, out_path):
    hs = summary_by_seed[summary_by_seed["condition_name"] == "hard_swap"].copy()
    hs["boundary"] = hs["boundary"].astype(int)
    plt.figure(figsize=(7, 4))
    sns.lineplot(data=hs, x="boundary", y="delta_accuracy_vs_noswap", hue="seed", marker="o", alpha=0.6)
    plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
    plt.title("Boundary Sensitivity: ΔAccuracy vs No-Swap")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def plot_validity_metrics(summary_by_seed, out_path):
    hs = summary_by_seed[summary_by_seed["condition_name"] == "hard_swap"].copy()
    hs["boundary"] = hs["boundary"].astype(int)
    m = hs.groupby("boundary", as_index=False).agg(valid_rate=("valid_rate", "mean"), malformed_rate=("malformed_rate", "mean"))
    m = m.melt(id_vars="boundary", var_name="metric", value_name="value")
    plt.figure(figsize=(7, 4))
    sns.lineplot(data=m, x="boundary", y="value", hue="metric", marker="o")
    plt.title("Boundary Sensitivity: Validity vs Malformed Rate")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()

plot_boundary_accuracy(summary_by_seed, plots_dir / "boundary_accuracy.png")
plot_delta_vs_noswap(summary_by_seed, plots_dir / "boundary_delta.png")
plot_validity_metrics(summary_by_seed, plots_dir / "boundary_validity.png")


## Part 11. Export artifacts


In [ ]:
def save_sample_records(df, output_dir):
    df.to_csv(output_dir / "sample_records.csv", index=False)


def save_summary_tables(summary_by_seed, summary_across_seeds, boundary_stability, output_dir):
    summary_by_seed.to_csv(output_dir / "summary_by_seed.csv", index=False)
    summary_across_seeds.to_csv(output_dir / "summary_across_seeds.csv", index=False)
    boundary_stability.to_csv(output_dir / "boundary_rank_stability.csv", index=False)


def save_selected_boundaries(selected, output_dir):
    with open(output_dir / "selected_boundaries.json", "w", encoding="utf-8") as f:
        json.dump(selected, f, indent=2, ensure_ascii=False)


def save_run_manifest(cfg, output_dir, dataset_df, active_boundaries):
    manifest = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "mode": cfg.mode,
        "model_family": cfg.model_family,
        "target_model_name": cfg.target_model_name,
        "anchor_model_name": cfg.anchor_model_name,
        "seeds": cfg.seeds,
        "boundaries": active_boundaries,
        "upper_boundary_t": cfg.upper_boundary_t,
        "dataset": {
            "name": cfg.dataset_name,
            "config": cfg.dataset_config_name,
            "split": cfg.dataset_split,
            "language": cfg.language,
            "n_evaluated": int(len(dataset_df)),
        },
        "generation": {
            "max_new_tokens": cfg.max_new_tokens,
            "do_sample": cfg.do_sample,
            "temperature": cfg.temperature,
            "top_p": cfg.top_p,
            "num_beams": cfg.num_beams,
            "stop_strings": cfg.stop_strings,
        },
        "packages": {
            "python": sys.version,
            "pytorch": torch.__version__,
            "transformers": __import__("transformers").__version__,
            "datasets": __import__("datasets").__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
    }
    with open(output_dir / "run_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

save_sample_records(sample_records_df, output_dir)
save_summary_tables(summary_by_seed, summary_across_seeds, boundary_stability, output_dir)
save_selected_boundaries(selected_boundaries, output_dir)
save_run_manifest(cfg, output_dir, dataset_df, active_boundaries)
print("Artifacts saved under:", output_dir)


## Part 12. Final interpretation note

This notebook establishes a **Stage-1 boundary sensitivity map** for cross-lingual layer composition.

Interpret outputs as:
- reproducible variation induced by boundary choice,
- relative stability vs instability across boundaries,
- differences relative to no-swap baseline.

Do **not** interpret outputs as mechanism proof, causal evidence, or an "optimal boundary" claim.
This notebook is preparatory for later mechanistic analysis.

### TODO extension hooks
- Add second backbone family (e.g., Llama 3.2-1B-Instruct).
- Add hidden-state extraction for selected boundaries.
- Add separability/similarity probes.
- Add activation patching/interventions.
- Add generic corruption controls.
